In [2]:
#!pip install -q peft bitsandbytes accelerate kagglehub librosa scikit-learn tqdm
#!pip uninstall -y transformers tokenizers librosa
#!pip install -U transformers accelerate
#!pip install -U librosa soundfile soxr

In [3]:
%pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.1 MB/s  0:00:01


In [4]:
# Import model quicker
#from huggingface_hub import login
#from google.colab import userdata

#login(token=userdata.get(""))

In [5]:
# Imports
import os
import gc
import re
import time
import random
import numpy as np
import pandas as pd
import torch
import librosa
import glob
from pathlib import Path
from google.colab import files
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
from transformers import (AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig)
from peft import (prepare_model_for_kbit_training, LoraConfig, get_peft_model)


In [6]:
# Setting all seed options for reproducibility
SEED = 42

random.seed(SEED) # Python
np.random.seed(SEED) # NumPy
torch.manual_seed(SEED) # PyTorch

torch.cuda.manual_seed(SEED) # Cuda
torch.cuda.manual_seed_all(SEED) # Cuda

os.environ["PYTHONHASHSEED"] = str(SEED) # More Python

In [7]:
import kagglehub
ravdess = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.


In [8]:
# Mapping the emotion labels
emotions = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

# Use the first 4 actors for evaluation, the remaining 20 actors are for finetuning the model
actors_val = list(range(1,5))
actors_train = list(range(5, 25))

# Searching the specific paths
audio_files = glob.glob(
    os.path.join(ravdess, "**", "*.wav"),
    recursive=True
)

records = []
seen = set()  # Use set to make sure there are no duplicates

for filepath in audio_files:

    # File name should look like:"03-01-06-01-02-01-12.wav"
    filename = os.path.basename(filepath)
    parts = filename.replace(".wav", "").split("-")

    if len(parts) != 7:
        continue

    # Relevant filename identifiers
    modality = parts[0]
    channel = parts[1]
    emotion_code = parts[2]
    actor = int(parts[6])

    # Modality should be 03 = Audio-only
    # Channel should be 01 = speech
    if modality != "03" or channel != "01":
        continue

    # To make the valid emotions are used
    if emotion_code not in emotions:
        continue

    # To make sure there are no duplicates
    if filename in seen:
        continue
    seen.add(filename)

    records.append({
        "path": filepath,
        "filename": filename,
        "emotion": emotions[emotion_code],
        "emotion_code": emotion_code,
        "actor": actor})

df = pd.DataFrame(records)

df["split"] = df["actor"].apply(
    lambda x: "evaluation"
    if x in actors_val
    else "train")

train_df = df[df["split"] == "train"].reset_index(drop=True)
eval_df = df[df["split"] == "evaluation"].reset_index(drop=True)


In [9]:
# Importhing the Qwen model --> https://huggingface.co/Qwen/Qwen2-Audio-7B-Instruct
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16)


model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True)


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/638k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

In [10]:
# Disable cache during training
model.config.use_cache = False

# Prepare the 4-bit model for LoRA training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Find attention projection layers ONLY inside the audio encoder
target_modules = [
    name
    for name, module in model.named_modules()
    if (
        name.startswith("model.audio_tower.")
        and name.endswith((".q_proj", ".k_proj", ".v_proj", ".o_proj"))
    )
]

print("Number of target modules:", len(target_modules))
print("\nFirst 10 target modules:")
for name in target_modules[:10]:
    print(name)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)

# Display trainable parameters
model.print_trainable_parameters()

# Verify that trainable parameters belong only to the audio encoder
trainable_names = [
    name for name, param in model.named_parameters()
    if param.requires_grad
]

print("\nTrainable parameter examples:")
for name in trainable_names[:15]:
    print(name)

assert all("audio_tower" in name for name in trainable_names), (
    "Unexpected trainable parameters outside the audio encoder!"
)

print("\nLoRA successfully attached to the audio encoder!")

Number of target modules: 96

First 10 target modules:
model.audio_tower.layers.0.self_attn.k_proj
model.audio_tower.layers.0.self_attn.v_proj
model.audio_tower.layers.0.self_attn.q_proj
model.audio_tower.layers.1.self_attn.k_proj
model.audio_tower.layers.1.self_attn.v_proj
model.audio_tower.layers.1.self_attn.q_proj
model.audio_tower.layers.2.self_attn.k_proj
model.audio_tower.layers.2.self_attn.v_proj
model.audio_tower.layers.2.self_attn.q_proj
model.audio_tower.layers.3.self_attn.k_proj
trainable params: 1,966,080 || all params: 8,399,060,992 || trainable%: 0.0234

Trainable parameter examples:
base_model.model.model.audio_tower.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.audio_tower.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.audio_tower.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.audio_tower.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.audio_tower.layers.0.self_attn.q_proj.lo

In [11]:
# The task given to QWEN
instruction = (
    "Listen carefully to the speaker's voice and identify the emotion"
    "which the speaker expresses through their tone, pitch, energy ect. "
    "Classify the emotion based on the voice, not the meaning of the words. "
    "Choose exactly ONE of these eight categories: "
    "neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
    "Respond with the emotion label only. Do not provide an explanation.")

In [12]:
# Configurations
epochs = 1
learning_rate = 1e-4
gradient_steps = 8
max_grad = 1.0
w_decay = 0.01

# Making sure QWEN is ready for training
model.config.use_cache = False
model.train()

# Only optimize LoRA parameters
trainable_parameters = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable_parameters, lr= learning_rate, weight_decay= w_decay)

# Shuffle training data for reproducibility
training_data = train_df.sample(frac=1, random_state= SEED).reset_index(drop=True)

sampling_rate = processor.feature_extractor.sampling_rate
audio_token_id = processor.tokenizer.convert_tokens_to_ids("<|AUDIO|>")


In [13]:
def prepare_training_example(sample):

    audio, sr = librosa.load(sample["path"], sr=sampling_rate)
    emotion = str(sample["emotion"]).lower().strip()

    # Conversation structure of QWEN
    prompt_conversation = [{
        "role": "user",
        "content": [
            {"type": "audio"},
            {"type": "text", "text": instruction}
        ]
    }]

    prompt_text = processor.apply_chat_template(prompt_conversation,
        add_generation_prompt=True, tokenize=False)

    full_conversation = prompt_conversation + [{
        "role": "assistant",
        "content": emotion
    }]

    full_text = processor.apply_chat_template(full_conversation,
        add_generation_prompt=False, tokenize=False)

    inputs = processor(text=full_text, audio=audio, sampling_rate=sr,
        return_tensors="pt", padding=True)

    input_ids = inputs["input_ids"]

    # Tokenise the prompt before audio expansion
    prompt_ids = processor.tokenizer(prompt_text, add_special_tokens=False,
        return_tensors="pt")["input_ids"]

    # Find the original audio placeholder
    placeholder_positions = (prompt_ids[0] == audio_token_id).nonzero(as_tuple=True)[0]

    placeholder_position = placeholder_positions[0].item()

    # Find the expanded audio tokens in the processed input
    audio_positions = (input_ids[0] == audio_token_id).nonzero(as_tuple=True)[0]

    expanded_audio_count = len(audio_positions)

    # Calculate the correct assistant-answer boundary
    prompt_length = (placeholder_position + expanded_audio_count
        + (prompt_ids.shape[1] - placeholder_position - 1))

    # Create labels: ignore everything before the answer
    labels = input_ids.clone()
    labels[:, :prompt_length] = -100

    if "attention_mask" in inputs:
        labels[inputs["attention_mask"] == 0] = -100

    # Verify the target contains a valid emotion
    target_tokens = labels[0][labels[0] != -100]

    target_text = processor.tokenizer.decode(
        target_tokens, skip_special_tokens=False)

    # Move inputs and labels to GPU
    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()}

    labels = labels.to(device)

    return inputs, labels, target_text

In [14]:
# Training
training_log = []
global_step = 0
start_time = time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(epochs):

    epoch_losses = []

    progress = tqdm(
        enumerate(training_data.iterrows()),
        total=len(training_data),
        desc=f"Epoch {epoch + 1}/{epochs}")

    for index, (_, sample) in progress:

        try:
            inputs, labels, target_text = prepare_training_example(sample)

            # Forward pass
            outputs = model(**inputs, labels=labels)
            loss = outputs.loss

            if not torch.isfinite(loss):
                raise ValueError(
                    f"Non-finite loss for {sample['filename']}"
                )

            raw_loss = loss.item()
            epoch_losses.append(raw_loss)

            # Backward pass
            (loss / gradient_steps).backward()

            # Update parameters every N recordings
            if (index + 1) % gradient_steps == 0:

                torch.nn.utils.clip_grad_norm_(
                    trainable_parameters, max_grad
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

            # Progress information
            average_loss = sum(epoch_losses) / len(epoch_losses)

            elapsed = time.time() - start_time
            recordings_done = index + 1

            estimated_total = (
                elapsed / recordings_done * len(training_data)
            )

            remaining = estimated_total - elapsed

            progress.set_postfix({
                "actor": sample["actor"],
                "emotion": sample["emotion"],
                "loss": f"{raw_loss:.4f}",
                "avg_loss": f"{average_loss:.4f}",
                "remaining": f"{remaining / 60:.1f}m"
            })

            del inputs, labels, outputs, loss

        except Exception as e:
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError(
                f"Training failed at recording "
                f"{sample['filename']}: {e}"
            ) from e

    # Handle any remaining accumulated gradients
    if len(training_data) % gradient_steps != 0:

        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_grad)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    print(f"\nEpoch {epoch + 1} completed!")
    print(f"Average training loss: {sum(epoch_losses) / len(epoch_losses):.4f}")
    print(f"Optimizer updates: {global_step}")

print("Done!")

Epoch 1/1:   0%|          | 0/1200 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



Epoch 1 completed!
Average training loss: 0.2413
Optimizer updates: 150
Done!


In [15]:
OUTPUT_CSV = "/content/ravdess_finetuned_predictions.csv"
# Start a fresh CSV
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

In [16]:
emotion_labels = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]

def extract_emotion(response):
    response = response.lower().strip()

    for emotion in emotion_labels:
        if re.search(r"\b" + re.escape(emotion) + r"\b", response):
            return emotion

    return "invalid"

In [17]:
def predict_emotion(audio_path, instruction):
    # Preprocesses the audio so QWEN can understand it
    audio, sr = librosa.load(audio_path, sr=processor.feature_extractor.sampling_rate)

    # Conversation structure of QWEN
    conversation = [
        {"role": "user",
          "content": [
                {"type": "audio"},
                {"type": "text", "text": instruction}
            ]}
    ]

    # Let QWEN generate the speaker's emotion
    text = processor.apply_chat_template(conversation, add_generation_prompt=True,
        tokenize=False)

    inputs = processor(text=text, audio=audio, sampling_rate=sr,
        return_tensors="pt", padding=True)

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=32,
            do_sample=False)

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

    response = processor.batch_decode(generated_ids, skip_special_tokens=True,
        clean_up_tokenization_spaces=False)[0]

    return extract_emotion(response), response

In [18]:
# Preparing the fine-tuned model for evaluation mode
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True


In [19]:
# Evaluating the finetuned model +
# Show live progress information to know how far along we are 🫡
results = []
start_time = time.time()

print("Start finetuned model evaluation")

progress = tqdm(eval_df.iterrows(), total=len(eval_df),
    desc="Evaluating RAVDESS", unit="recording",  dynamic_ncols=True)

for i, (_, sample) in enumerate(progress, start=1):
    recording_start = time.time()

    try:
        predicted_emotion, raw_response = predict_emotion(sample["path"], instruction)
        result = {
            "filename": sample["filename"],
            "actor": sample["actor"],
            "actual_emotion": sample["emotion"],
            "predicted_emotion": predicted_emotion}

    except Exception as e:
        print(f"\nError processing {sample['filename']}: {e}")

    results.append(result)

    # Save immediately after each recording, so in case of error not all progress is lost
    pd.DataFrame([result]).to_csv(OUTPUT_CSV, mode="a",
        header=not os.path.exists(OUTPUT_CSV), index=False)

    elapsed = time.time() - start_time
    recording_time = time.time() - recording_start

    progress.set_postfix({
        "Actor": sample["actor"],
        "Actual": sample["emotion"],
        "Predicted": result["predicted_emotion"],
        "Last": f"{recording_time:.1f}s"})

progress.close()

# Download the csv after finishing
files.download("/content/ravdess_finetuned_predictions.csv")

Start finetuned model evaluation


Evaluating RAVDESS:   0%|          | 0/240 [00:00<?, ?recording/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Second use scenario
Asking QWEN without specification to classify the speakers emotion


In [20]:
OUTPUT_CSV = "/content/ravdess_finetuned_predictions_2.csv"
# Start a fresh CSV
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

In [21]:
instruction_2 = (
    "Listen carefully to the speaker's voice and identify the emotion. "
    "Choose exactly ONE of these eight categories: "
    "neutral, calm, happy, sad, angry, fearful, disgust, surprised. "
    "Respond with the emotion label only. Do not provide an explanation.")

In [22]:
# Evaluating the finetuned model +
# Show live progress information to know how far along we are 🫡
results = []
start_time = time.time()

print("Start finetuned model evaluation")

progress = tqdm(eval_df.iterrows(), total=len(eval_df),
    desc="Evaluating RAVDESS", unit="recording",  dynamic_ncols=True)

for i, (_, sample) in enumerate(progress, start=1):
    recording_start = time.time()

    try:
        predicted_emotion, raw_response = predict_emotion(sample["path"], instruction_2)
        result = {
            "filename": sample["filename"],
            "actor": sample["actor"],
            "actual_emotion": sample["emotion"],
            "predicted_emotion": predicted_emotion}

    except Exception as e:
        print(f"\nError processing {sample['filename']}: {e}")

    results.append(result)

    # Save immediately after each recording, so in case of error not all progress is lost
    pd.DataFrame([result]).to_csv(OUTPUT_CSV, mode="a",
        header=not os.path.exists(OUTPUT_CSV), index=False)

    elapsed = time.time() - start_time
    recording_time = time.time() - recording_start

    progress.set_postfix({
        "Actor": sample["actor"],
        "Actual": sample["emotion"],
        "Predicted": result["predicted_emotion"],
        "Last": f"{recording_time:.1f}s"})

progress.close()

# Download the csv after finishing
files.download("/content/ravdess_finetuned_predictions_2.csv")

Start finetuned model evaluation


Evaluating RAVDESS:   0%|          | 0/240 [00:00<?, ?recording/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>